# Power Outage Forecasting
This notebook is a demo for how to load, process, train on, and produce prediction files with the provided dataset.
You should not be confined by this notebook, but highly encouraged to produce your own pipeline or models based on what you learned from the course.

The format of submission files, however, is VERY strict. Student have to follow the provided tempaltes and the submitted files must pass the sanity tests in the last section.

Note the test files loaded in this notebook contain random noise. They only serve the purpose to dmostrate how evalution will be done. The actual test file has been hidden.

It is encouraged to run the starter notebook in Google Colab for easy of debug and implementation. You could also use your local Python notebook.
There is a chance that you run into out of memory when running this demo notebook in Colab free tier. In that case, you could either apply for free Google Colab Pro subscription as student and turn on high RAM, or only run one of the three models (all of the SARIMAX models, Seq2seq model for 24h, or Seq2seq model for 48h) included in the notebook.

## ☁️ Colab / Local 环境自动配置

**运行下面这个 cell 即可**，它会自动检测你在 Colab 还是本地，完成所有配置：
- Colab：挂载 Google Drive、创建 `.env`、设置数据路径
- 本地：跳过，什么都不做

> **Colab 用户请先做好以下准备：**
> 1. 将 `train.nc` 上传到你的 Google Drive 某个文件夹（如 `My Drive/MLPS_Data/train.nc`）
> 2. 准备好你的 wandb 信息（用户名、API Key、Team 名称）
> 3. 运行下方 cell，按提示操作

In [ ]:
# ============================================================
# Environment Auto-Configuration (Colab / Local)
# ============================================================
# This cell detects your environment and sets everything up.
# - Colab: mounts Drive, creates .env, locates data
# - Local: does nothing (you already have .env and data/)
# ============================================================

import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("☁️  Google Colab detected!\n")
    
    # --- 1. Mount Google Drive ---
    from google.colab import drive
    drive.mount("/content/drive")
    print()
    
    # --- 2. Clone repo (if not already) ---
    REPO_DIR = "/content/MLPS_Final_Project"
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        os.system("git clone https://github.com/wangruig-lang/MLPS_Final_Project.git " + REPO_DIR)
    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    
    # --- 3. Locate train.nc on Google Drive ---
    # Common locations to search
    DRIVE_SEARCH_PATHS = [
        "/content/drive/MyDrive/MLPS_Data/train.nc",
        "/content/drive/MyDrive/MLPS_Final_Project/data/train.nc",
        "/content/drive/MyDrive/train.nc",
        "/content/drive/MyDrive/data/train.nc",
    ]
    
    train_nc_path = None
    for p in DRIVE_SEARCH_PATHS:
        if os.path.exists(p):
            train_nc_path = p
            break
    
    if train_nc_path:
        print(f"\n✓ Found train.nc: {train_nc_path}")
    else:
        print("\n⚠️  train.nc not found in common Drive locations.")
        print("   Searched:", DRIVE_SEARCH_PATHS)
        train_nc_path = input("   Please enter the full path to train.nc on your Drive: ").strip()
        if not os.path.exists(train_nc_path):
            print(f"   ✗ File not found: {train_nc_path}")
            print("   You can continue, but data loading will fail later.")
    
    # Symlink train.nc into data/
    os.makedirs("data", exist_ok=True)
    local_train = "data/train.nc"
    if train_nc_path and os.path.exists(train_nc_path) and not os.path.exists(local_train):
        os.symlink(train_nc_path, local_train)
        print(f"✓ Linked: {local_train} → {train_nc_path}")
    
    # --- 4. Create .env file interactively ---
    if not os.path.exists(".env"):
        print("\n--- Wandb Configuration ---")
        print("(Leave blank to skip wandb logging)\n")
        wb_user = input("Your wandb username: ").strip()
        wb_key = input("Your wandb API key: ").strip()
        wb_entity = input("Your wandb team name: ").strip()
        
        with open(".env", "w") as f:
            f.write(f"WANDB_USERNAME={wb_user}\n")
            f.write(f"WANDB_API_KEY={wb_key}\n")
            f.write(f"WANDB_ENTITY={wb_entity}\n")
        
        if wb_user and wb_key and wb_entity:
            print("✓ .env created with wandb credentials")
        else:
            print("✓ .env created (wandb disabled — credentials incomplete)")
    else:
        print("\n✓ .env already exists")
    
    # --- 5. Enable GPU info ---
    if os.system("nvidia-smi > /dev/null 2>&1") == 0:
        print("\n✓ GPU available!")
        os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
    else:
        print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU for faster training")
    
    print("\n" + "="*50)
    print("✓ Colab setup complete! Continue running the next cells.")
    print("="*50)

else:
    print("💻 Local environment detected. No extra setup needed.")
    print("   Make sure you have:")
    print("   - data/train.nc downloaded")
    print("   - .env file configured (cp .env.example .env)")

## Prerequisites & Installation

Before running this demo notebook, ensure you have the required packages installed:

- `xarray` - for reading NetCDF files
- `netCDF4` - backend for xarray
- `numpy` - numerical operations
- `pandas` - data manipulation
- `matplotlib` - visualization
- `torch` - deep learning (PyTorch)
- `statsmodels` - SARIMAX models
- `tqdm` - progress bars


In [ ]:
# Install dependencies
import subprocess, sys, platform

IN_COLAB = "google.colab" in sys.modules

def is_conda():
    return "conda" in sys.prefix or "anaconda" in sys.prefix

if IN_COLAB:
    # Colab already has numpy, pandas, matplotlib, torch, tqdm
    # Just install the missing ones
    print("☁️  Colab: Installing missing packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "xarray", "netCDF4", "statsmodels", "wandb", "python-dotenv"])
    print("✓ All packages installed.")

elif is_conda():
    print("Conda environment detected.")
    print("Installing netCDF4 and PyTorch via conda (this may take a minute)...")
    conda_pkgs = ["netcdf4", "pytorch"]
    if platform.system() != "Darwin":
        conda_pkgs.append("cpuonly")
    subprocess.check_call(
        ["conda", "install", "-y", "-c", "conda-forge", "-c", "pytorch"] + conda_pkgs
    )
    print("Done. Installing remaining packages via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("✓ All packages installed.")

else:
    # Pure pip environment
    print("Installing dependencies via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "--index-url", "https://download.pytorch.org/whl/cpu"])
    print("✓ All packages installed.")

In [13]:
# Check dependencies from requirements.txt
import sys

# Map package names to their import names (only for cases where they differ)
IMPORT_NAME_MAP = {
    "python-dotenv": "dotenv",
    "netCDF4": "netCDF4",
}

missing_packages = []
with open("requirements.txt") as f:
    for line in f:
        package = line.strip()
        if not package or package.startswith("#"):
            continue
        import_name = IMPORT_NAME_MAP.get(package, package)
        try:
            __import__(import_name)
            print(f"✓ {package} is installed")
        except ImportError:
            print(f"✗ {package} is NOT installed")
            missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
    print("Run: pip install -r requirements.txt")
else:
    print("\n✓ All required packages are installed!")

✓ numpy is installed
✓ pandas is installed
✓ xarray is installed
✓ netCDF4 is installed
✓ matplotlib is installed
✓ statsmodels is installed
✓ tqdm is installed
✓ wandb is installed
✓ python-dotenv is installed

✓ All required packages are installed!


## 1. Configuration & Setup

Define all meta variables and parameters at the start for easy configuration.

In [14]:
# ============== DATA SETUP ==============
# train.nc is too large for GitHub. Please download it manually:
#   1. Download train.nc from the shared Google Drive / Canvas
#   2. Place it in the data/ folder: data/train.nc
# The other data files (test_24h_demo.nc, test_48h_demo.nc) are already in data/
# ==========================================

# ============== META VARIABLES ==============
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Data paths
DATA_DIR = "data/"
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.nc")

# The test_24h_demo.nc and test_48h_demo.nc are just demo files, so the outages in those files are just noise
# TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h.nc")
# TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h.nc")
TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h_demo.nc")
TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h_demo.nc")

# Model parameters
VALIDATION_SPLIT = 0.2  # Use last 20% of training data for validation

# SARIMAX parameters
SARIMAX_ORDER = (1, 0, 1)  # (p, d, q)

# Seq2Seq parameters
SEQ_LEN = 24       # Lookback window (hours) for the seq2seq model
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
HIDDEN_DIM = 64
NUM_LAYERS = 1

# Set device for PyTorch
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============== WANDB SETUP ==============
import wandb
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()  # Load .env file

WANDB_USERNAME = os.getenv("WANDB_USERNAME")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")  # Team name on wandb

if not WANDB_USERNAME or not WANDB_API_KEY or not WANDB_ENTITY:
    print("⚠️  WANDB credentials not fully configured in .env file!")
    print("   Required: WANDB_USERNAME, WANDB_API_KEY, WANDB_ENTITY")
    print("   Copy .env.example to .env and fill in your credentials.")
    print("   Notebook will still run, but experiments won't be logged to W&B.\n")
    WANDB_ENABLED = False
else:
    wandb.login(key=WANDB_API_KEY)
    run_name = f"{WANDB_USERNAME}_{datetime.now().strftime('%m%d_%H%M%S')}"
    wandb.init(
        project="MLPS-Power-Outage",
        entity=WANDB_ENTITY,
        name=run_name,
        config={
            "random_seed": RANDOM_SEED,
            "validation_split": VALIDATION_SPLIT,
            "sarimax_order": SARIMAX_ORDER,
            "seq_len": SEQ_LEN,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "device": DEVICE,
        },
    )
    WANDB_ENABLED = True
    print(f"✓ W&B initialized: run={run_name}, entity={WANDB_ENTITY}")

print(f"\nConfiguration loaded successfully!")
print(f"Random Seed: {RANDOM_SEED}")
print(f"Device: {DEVICE}")
print(f"Data Directory: {DATA_DIR}")
print(f"Results Directory: {RESULTS_DIR}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/gongwangrui/.netrc


CommError: Error uploading run: returned error 404: {"data":{"upsertBucket":null},"errors":[{"message":"entity your_team_name not found during upsertBucket","path":["upsertBucket"]}]}

## 2. Data Loading

Load the NetCDF datasets and explore their structure.

In [ ]:
# Load datasets
ds_train = xr.open_dataset(TRAIN_PATH)
ds_test_24h = xr.open_dataset(TEST_24H_PATH)
ds_test_48h = xr.open_dataset(TEST_48H_PATH)

In [ ]:
# Extract basic information
train_timestamps = pd.to_datetime(ds_train.timestamp.values)
locations = list(ds_train.location.values)
weather_features = list(ds_train.feature.values) if 'feature' in ds_train.dims else []

print(f"Training Period: {train_timestamps.min()} to {train_timestamps.max()}")
print(f"Number of Timestamps: {len(train_timestamps)}")
print(f"Number of Locations: {len(locations)}")
print(f"Locations: {locations}")
print(f"\nWeather Features ({len(weather_features)}): {weather_features}")

# Extract outage data
outage_data = ds_train.out.transpose("timestamp", "location").values.astype(float)
print(f"\nOutage Data Shape: {outage_data.shape} (timestamps x locations)")
print(f"Outage Statistics:")
print(f"  Mean: {np.nanmean(outage_data):.2f}")
print(f"  Std: {np.nanstd(outage_data):.2f}")
print(f"  Min: {np.nanmin(outage_data):.2f}")
print(f"  Max: {np.nanmax(outage_data):.2f}")

In [ ]:
# Load test datasets
print("Loading test datasets...")

test_24h_timestamps = ds_test_24h.timestamp.values
test_48h_timestamps = ds_test_48h.timestamp.values

print(f"✓ Test 24h: {len(test_24h_timestamps)} timestamps")
print(f"✓ Test 48h: {len(test_48h_timestamps)} timestamps")

print(f"\nTesting Period (24h): {test_24h_timestamps.min()} to {test_24h_timestamps.max()}")
print(f"Testing Period (48h): {test_48h_timestamps.min()} to {test_48h_timestamps.max()}")

In [ ]:
# sanity check on outage being positive and smaller than total tracked household
print(bool((ds_train.out <= ds_train.tracked).all()))
print(bool((ds_train.out>=0).all()))
print(bool((ds_train.tracked>=0).all()))

## 3. Exploratory Data Analysis

EDA is important for you to understand the data and the problem. Here I just show some basics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) Compute population per county
pop_by_loc = ds_train.tracked.mean(dim="timestamp")

# 2) Get top 10 locations by population
top5_locs = (
    pop_by_loc
    .sortby(pop_by_loc, ascending=False)
    .isel(location=slice(0, 5))
    .location
    .values
)

print("Top 5 locations by population:", [str(x) for x in top5_locs])

# 3) Plot their outages over time
fig, ax = plt.subplots(figsize=(15, 6))

for loc in top5_locs:
    outages = ds_train.out.sel(location=loc).values
    ax.plot(train_timestamps, outages, label=str(loc), alpha=0.8, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Power Outages", fontsize=12)
ax.set_title("Power Outages Over Time — Top 10 Counties by Population",
             fontsize=14, fontweight="bold")
ax.legend(ncol=2, fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Get the feature index for t2m
t2m_idx = weather_features.index('t2m')

fig, ax = plt.subplots(figsize=(15, 6))

# Plot t2m for the top 5 counties
for loc in top5_locs:
    # Extract t2m data for this location: (timestamp, feature) -> select t2m
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    ax.plot(train_timestamps, t2m_data, label=str(loc), alpha=0.8, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Temperature at 2m (K)", fontsize=12)
ax.set_title("Temperature (t2m) Over Time — Top 5 Counties by Population",
                fontsize=14, fontweight="bold")
ax.legend(ncol=2, fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print some statistics
print("\nt2m Statistics for Top 5 Counties:")
for loc in top5_locs:
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    print(f"  {loc}: Mean={np.mean(t2m_data):.2f}K, "
            f"Min={np.min(t2m_data):.2f}K, Max={np.max(t2m_data):.2f}K")

In [ ]:
print("Correlation Analysis: t2m (Temperature) vs. Power Outages\n")
print("="*70)

correlations = {}

for loc in top5_locs:
    # Get t2m data
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    # Get outage data
    outage_data_loc = ds_train.out.sel(location=loc).values
    
    # Remove NaN values for correlation calculation
    valid_mask = ~(np.isnan(t2m_data) | np.isnan(outage_data_loc))
    t2m_clean = t2m_data[valid_mask]
    outage_clean = outage_data_loc[valid_mask]
    
    # Calculate Pearson correlation
    if len(t2m_clean) > 1:
        correlation = np.corrcoef(t2m_clean, outage_clean)[0, 1]
        correlations[str(loc)] = correlation
        
        direction = "Positive" if correlation > 0 else "Negative"
        
        print(f"County {loc}:")
        print(f"  Correlation: {correlation:+.4f} ({direction})")
        print(f"  Sample size: {len(t2m_clean)} valid data points")
        print()
    else:
        print(f"County {loc}: Insufficient data for correlation")
        correlations[str(loc)] = np.nan

# Visualize correlations
fig, ax = plt.subplots(figsize=(10, 6))

counties_list = list(correlations.keys())
corr_values = list(correlations.values())
colors = ['red' if c < 0 else 'blue' for c in corr_values]

bars = ax.bar(counties_list, corr_values, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('County', fontsize=12)
ax.set_ylabel('Correlation Coefficient', fontsize=12)
ax.set_title('Correlation between Temperature (t2m) and Power Outages\nTop 5 Counties by Population', 
                fontsize=14, fontweight='bold')
ax.set_ylim(-1, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, corr_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + (0.05 if height > 0 else -0.05),
            f'{val:.3f}', ha='center', va='bottom' if height > 0 else 'top', 
            fontsize=10, fontweight='bold')


## 4. Data Preparation

Split the training data into train and validation sets for model selection.

In [ ]:
# Create validation split (temporal split)
n_timestamps = len(train_timestamps)
split_idx = int(n_timestamps * (1 - VALIDATION_SPLIT))

# Split the dataset
ds_train_sub = ds_train.isel(timestamp=slice(0, split_idx))
ds_val = ds_train.isel(timestamp=slice(split_idx, None))

train_sub_timestamps = pd.to_datetime(ds_train_sub.timestamp.values)
val_timestamps = pd.to_datetime(ds_val.timestamp.values)

print(f"Training Subset: {len(train_sub_timestamps)} timestamps")
print(f"  Period: {train_sub_timestamps.min()} to {train_sub_timestamps.max()}")
print(f"\nValidation Set: {len(val_timestamps)} timestamps")
print(f"  Period: {val_timestamps.min()} to {val_timestamps.max()}")

# Store validation ground truth for later evaluation
val_truth = ds_val.out.transpose("timestamp", "location").values.astype(float)

In [ ]:
# Prepare ground truth for last 24 and 48 hours to align with test set
val_truth_24h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 24)).values.astype(float)
val_truth_48h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 48)).values.astype(float)

print(f"\Validation set shapes:")
print(f"  24h: {val_truth_24h.shape}")
print(f"  48h: {val_truth_48h.shape}")

In [ ]:
test_24h_truth = ds_test_24h.out.transpose("timestamp", "location").values.astype(float)
test_48h_truth = ds_test_48h.out.transpose("timestamp", "location").values.astype(float)


print(f"\nTest shapes:")
print(f"  24h: {test_24h_truth.shape}")
print(f"  48h: {test_48h_truth.shape}")

## 5. Model Structure and Functions

We'll define two types of models:
1. **SARIMAX** - Statistical time series model (per-county)
2. **Seq2Seq LSTM** - Deep learning model (shared across counties but different for two horizons)

### 5.1 SARIMAX

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def safe_fit_sarimax(y, order=(1, 0, 1)):
    """Safely fit SARIMAX model with error handling."""
    y = np.asarray(y, dtype=float).flatten()
    if len(y) < 8 or np.allclose(y, y[0]):
        return None
    try:
        model = SARIMAX(y, order=order, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        return res
    except Exception as e:
        print(f"  Warning: SARIMAX fit failed - {str(e)[:50]}")
        return None


### 5.2 Seq2Seq

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import time

# Set random seed for PyTorch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# Define Seq2Seq Model
class SimpleSeq2Seq(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, horizon=24):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_dim, horizon)
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        _, (h, _) = self.lstm(x)
        h_last = h[-1]  # (batch, hidden_dim)
        return self.head(h_last)  # (batch, horizon)

print(f"Device: {DEVICE}")

In [ ]:
# Utility functions for Seq2Seq
def z_normalize_fit(arr):
    """Compute mean and std for normalization."""
    mu = np.nanmean(arr, axis=0)
    sd = np.nanstd(arr, axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    return mu, sd

def z_normalize_apply(arr, mu, sd):
    """Apply normalization with precomputed mean and std."""
    return (arr - mu) / sd

def build_sliding_windows(X_loc, y_loc, seq_len, horizon):
    """
    Build sliding windows for one location.
    X_loc: (T, D) features
    y_loc: (T,) targets
    Returns: X_windows (N, seq_len, D), Y_windows (N, horizon)
    """
    N = len(y_loc) - seq_len - horizon + 1
    if N <= 0:
        return np.empty((0, seq_len, X_loc.shape[1]), dtype=float), np.empty((0, horizon), dtype=float)
    
    X_windows, Y_windows = [], []
    for i in range(N):
        X_windows.append(X_loc[i:i+seq_len])
        Y_windows.append(y_loc[i+seq_len:i+seq_len+horizon])
    
    return np.array(X_windows, dtype=float), np.array(Y_windows, dtype=float)

print("Helper functions defined.")

In [ ]:
# Prepare training data for Seq2Seq (using training subset)
def prepare_seq2seq_data(ds, seq_len, horizon):
    """
    Prepare data for Seq2Seq training.
    Features: [outage_scaled, weather_features_scaled]
    """
    y = ds.out.transpose("timestamp", "location").values.astype(float)  # (T, L)
    w = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)  # (T, L, F)
    T, L, F = w.shape
    
    # Compute global scalers
    y_mu, y_sd = z_normalize_fit(y.reshape(-1, 1))
    w_mu, w_sd = z_normalize_fit(w.reshape(-1, F))
    
    # Apply scaling
    y_scaled = z_normalize_apply(y.reshape(-1, 1), y_mu, y_sd).reshape(T, L)
    w_scaled = z_normalize_apply(w.reshape(-1, F), w_mu, w_sd).reshape(T, L, F)
    
    # Build windows for all locations
    input_dim = 1 + F  # outage + weather features
    X_list, Y_list = [], []
    
    for li in range(L):
        y_loc = y_scaled[:, li]  # (T,)
        w_loc = w_scaled[:, li, :]  # (T, F)
        X_loc = np.concatenate([y_loc.reshape(-1, 1), w_loc], axis=1)  # (T, 1+F)
        
        X_win, Y_win = build_sliding_windows(X_loc, y_loc, seq_len, horizon)
        if len(X_win) > 0:
            X_list.append(X_win)
            Y_list.append(Y_win)
    
    X = np.concatenate(X_list, axis=0) if X_list else np.empty((0, seq_len, input_dim))
    Y = np.concatenate(Y_list, axis=0) if Y_list else np.empty((0, horizon))
    
    scalers = {"y_mu": y_mu, "y_sd": y_sd, "w_mu": w_mu, "w_sd": w_sd}
    return X, Y, input_dim, scalers

# Use validation set length as horizon for training
val_horizon = len(val_timestamps)
print(f"Preparing Seq2Seq training data with horizon={val_horizon}...")

X_train, Y_train, input_dim, seq2seq_scalers = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, val_horizon)

print(f"Training windows created: {X_train.shape[0]} samples")
print(f"Input shape: {X_train.shape} (samples, seq_len, features)")
print(f"Output shape: {Y_train.shape} (samples, horizon)")
print(f"Input dimension: {input_dim}")

In [ ]:
# Train Seq2Seq model with optional validation RMSE monitoring
def train_seq2seq(X, Y, input_dim, horizon, epochs=5, batch_size=64, lr=1e-3,
                  X_val=None, Y_val=None, scalers=None):
    """
    Train Seq2Seq model.
    
    If X_val / Y_val are provided, computes and logs validation RMSE each epoch.
    If scalers dict (with y_mu, y_sd) is given, RMSE is reported in original scale
    (matching the competition metric).
    """
    if len(X) == 0:
        print("No training data available!")
        return None
    
    # Create dataset and dataloader
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(Y, dtype=torch.float32)
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Prepare validation tensors (if provided)
    has_val = X_val is not None and Y_val is not None and len(X_val) > 0
    if has_val:
        X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        Y_val_t = torch.tensor(Y_val, dtype=torch.float32).to(DEVICE)
        # For inverse-scaling to original units
        if scalers is not None:
            y_mu = float(scalers["y_mu"])
            y_sd = float(scalers["y_sd"])
        else:
            y_mu, y_sd = 0.0, 1.0  # no inverse scaling
    
    # Initialize model
    model = SimpleSeq2Seq(
        input_dim=input_dim,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        horizon=horizon
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    best_val_rmse = float("inf")
    
    print(f"\nTraining Seq2Seq model for {epochs} epochs...")
    if has_val:
        print(f"  Validation samples: {len(X_val)} (RMSE in original scale)")
    
    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        epoch_loss = 0.0
        start_time = time.time()
        
        for xb, yb in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            epoch_loss += loss.item() * xb.size(0)
        
        epoch_loss /= len(dataset)
        train_rmse_scaled = float(np.sqrt(epoch_loss))
        train_rmse = train_rmse_scaled * y_sd if has_val else train_rmse_scaled
        elapsed = time.time() - start_time
        
        # ---- Validation RMSE (competition metric proxy) ----
        val_rmse = None
        if has_val:
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t)  # (N, horizon) in scaled space
                val_mse = criterion(val_pred, Y_val_t).item()
                # RMSE in original scale: sqrt(MSE) * y_sd
                val_rmse = float(np.sqrt(val_mse)) * y_sd
                if val_rmse < best_val_rmse:
                    best_val_rmse = val_rmse
        
        # ---- Logging ----
        log_msg = f"Epoch {epoch+1}/{epochs} - Train RMSE: {train_rmse:.4f}"
        if val_rmse is not None:
            log_msg += f" - Val RMSE: {val_rmse:.4f} (best: {best_val_rmse:.4f})"
        log_msg += f" - Time: {elapsed:.1f}s"
        print(log_msg)
        
        if WANDB_ENABLED:
            log_dict = {
                f"seq2seq_{horizon}h/train_loss": epoch_loss,
                f"seq2seq_{horizon}h/train_rmse": train_rmse,
                f"seq2seq_{horizon}h/epoch": epoch + 1,
                f"seq2seq_{horizon}h/epoch_time": elapsed,
            }
            if val_rmse is not None:
                log_dict[f"seq2seq_{horizon}h/val_rmse"] = val_rmse
                log_dict[f"seq2seq_{horizon}h/best_val_rmse"] = best_val_rmse
            wandb.log(log_dict)
    
    if has_val:
        print(f"  ✓ Best validation RMSE: {best_val_rmse:.4f}")
    
    return model

### 5.3 Prediction Functions

Define reusable prediction functions that will be used for both validation and test sets.

In [ ]:
# Consolidated prediction functions for reuse

def generate_sarimax_predictions(models_dict, locations, timestamps):
    """
    Generate SARIMAX predictions for given timestamps.
    Returns: DataFrame in long format (timestamp, location, pred)
    """
    rows = []
    n_steps = len(timestamps)
    
    for loc in locations:
        loc_str = str(loc)
        if loc_str in models_dict and models_dict[loc_str] is not None:
            try:
                pred = np.asarray(models_dict[loc_str].forecast(steps=n_steps), dtype=float)
                pred = np.clip(pred, 0, None)  # Non-negative constraint
            except:
                pred = np.zeros(n_steps)
        else:
            pred = np.zeros(n_steps)
        
        rows.append(pd.DataFrame({
            "timestamp": timestamps,
            "location": loc_str,
            "pred": pred
        }))
    
    return pd.concat(rows, ignore_index=True)

@torch.no_grad()
def generate_seq2seq_predictions(model, ds_train_data, scalers, horizon, timestamps, locations):
    """
    Generate Seq2Seq predictions for given horizon.
    Returns: DataFrame in long format (timestamp, location, pred)
    """
    if model is None:
        # Return zeros if model not available
        locs_repeated = np.repeat([str(loc) for loc in locations], horizon)
        ts_repeated = np.tile(timestamps, len(locations))
        return pd.DataFrame({
            "timestamp": ts_repeated,
            "location": locs_repeated,
            "pred": 0.0
        })
    
    model.eval()
    
    y = ds_train_data.out.transpose("timestamp", "location").values.astype(float)
    w = ds_train_data.weather.transpose("timestamp", "location", "feature").values.astype(float)
    T, L, F = w.shape
    
    # Apply scaling
    y_scaled = z_normalize_apply(y.reshape(-1, 1), scalers["y_mu"], scalers["y_sd"]).reshape(T, L)
    w_scaled = z_normalize_apply(w.reshape(-1, F), scalers["w_mu"], scalers["w_sd"]).reshape(T, L, F)
    
    predictions = []
    
    for li in range(L):
        if T < SEQ_LEN:
            pred_scaled = np.zeros(horizon)
        else:
            # Get last SEQ_LEN timesteps
            y_loc = y_scaled[:, li]
            w_loc = w_scaled[:, li, :]
            X_loc = np.concatenate([y_loc.reshape(-1, 1), w_loc], axis=1)
            X_in = X_loc[-SEQ_LEN:].reshape(1, SEQ_LEN, -1)
            
            # Predict
            X_tensor = torch.tensor(X_in, dtype=torch.float32).to(DEVICE)
            pred_scaled = model(X_tensor).cpu().numpy()[0]
        
        # Inverse scaling
        pred = (pred_scaled * scalers["y_sd"].flatten()[0]) + scalers["y_mu"].flatten()[0]
        pred = np.clip(pred, 0, None)  # Non-negative constraint
        predictions.append(pred)
    
    # Convert to DataFrame in long format
    preds_array = np.array(predictions).T  # (horizon, L)
    rows = []
    for i, loc in enumerate(locations):
        rows.append(pd.DataFrame({
            "timestamp": timestamps,
            "location": str(loc),
            "pred": preds_array[:, i]
        }))
    
    return pd.concat(rows, ignore_index=True)

### 5.4 Evaluation Metric Fucntions

In [ ]:
# Define evaluation metrics
def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# def mae(y_true, y_pred):
#     """Calculate Mean Absolute Error."""
#     y_true = np.asarray(y_true, dtype=float)
#     y_pred = np.asarray(y_pred, dtype=float)
#     return float(np.mean(np.abs(y_true - y_pred)))

## 6. Validation Set (24h and 48h)

Evaluate models on 24h and 48h prediction horizons separately, matching the test set structure.

In [ ]:
# Define horizons
horizon_24h_val = 24
horizon_48h_val = 48

# Extract validation timestamps for each horizon
val_timestamps_24h = val_timestamps[:24]
val_timestamps_48h = val_timestamps[:48]

### 6.1 Train Models

In [ ]:
# Train SARIMAX models per county on training subset
print("Training SARIMAX models (one per county)...\n")
sarimax_models = {}

# Notice we fixed SARIMAX_ORDER for all counties, however, one can easily tune it over validation set for each county.
for loc in locations:
    print(f"Fitting SARIMAX for {loc}...", end=" ")
    y_train = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()
    model = safe_fit_sarimax(y_train, order=SARIMAX_ORDER)
    sarimax_models[loc] = model
    if model is not None:
        print(f"✓ Success (AIC: {model.aic:.2f})")
    else:
        print("✗ Failed (will use zero baseline)")

print(f"\nSARIMAX training complete!")

In [ ]:
# Train Seq2Seq models per county on training subset
# Also prepare validation windows for in-training RMSE monitoring
print("Training SARIMAX models (one per time horizon for all counties)...\n")
print(f"\nValidation horizons:")
print(f"  24h: {len(val_timestamps_24h)} timestamps")
print(f"  48h: {len(val_timestamps_48h)} timestamps")

# Prepare validation windows from the held-out validation portion
# so we can monitor RMSE *during* training (not just after)
print("\nPreparing validation windows for in-training RMSE monitoring...")
X_val_24h, Y_val_24h, _, _ = prepare_seq2seq_data(ds_val, SEQ_LEN, horizon_24h_val)
X_val_48h, Y_val_48h, _, _ = prepare_seq2seq_data(ds_val, SEQ_LEN, horizon_48h_val)
print(f"  24h val windows: {X_val_24h.shape[0]}")
print(f"  48h val windows: {X_val_48h.shape[0]}")

# Train for 24h horizon
print(f"\n{'='*70}")
print("Training Seq2Seq for 24h horizon...")
print(f"{'='*70}")
X_train_24h, Y_train_24h, input_dim, seq2seq_scalers_24h = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, horizon_24h_val)
print(f"Training samples: {X_train_24h.shape[0]}")

seq2seq_model_24h_val = train_seq2seq(
    X_train_24h, Y_train_24h, input_dim, horizon_24h_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
    X_val=X_val_24h, Y_val=Y_val_24h, scalers=seq2seq_scalers_24h
)

# Train for 48h horizon
print(f"\n{'='*70}")
print("Training Seq2Seq for 48h horizon...")
print(f"{'='*70}")
X_train_48h, Y_train_48h, _, seq2seq_scalers_48h = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, horizon_48h_val)
print(f"Training samples: {X_train_48h.shape[0]}")

seq2seq_model_48h_val = train_seq2seq(
    X_train_48h, Y_train_48h, input_dim, horizon_48h_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
    X_val=X_val_48h, Y_val=Y_val_48h, scalers=seq2seq_scalers_48h
)

print("\n✓ Seq2Seq models trained for both horizons!")

### 6.2 Generate Predictions

In [ ]:
# Generate predictions using consolidated functions
print("Generating predictions for validation set...\n")

# SARIMAX predictions (same models for both horizons, just different forecast lengths)
print("1) SARIMAX predictions...")
sarimax_val_pred_24h_df = generate_sarimax_predictions(sarimax_models, locations, val_timestamps_24h)
sarimax_val_pred_48h_df = generate_sarimax_predictions(sarimax_models, locations, val_timestamps_48h)
print(f"   24h: {len(sarimax_val_pred_24h_df)} predictions")
print(f"   48h: {len(sarimax_val_pred_48h_df)} predictions")

# Seq2Seq predictions
print("\n2) Seq2Seq predictions...")
seq2seq_val_pred_24h_df = generate_seq2seq_predictions(
    seq2seq_model_24h_val, ds_train_sub, seq2seq_scalers_24h, 
    horizon_24h_val, val_timestamps_24h, locations
)
seq2seq_val_pred_48h_df = generate_seq2seq_predictions(
    seq2seq_model_48h_val, ds_train_sub, seq2seq_scalers_48h,
    horizon_48h_val, val_timestamps_48h, locations
)
print(f"   24h: {len(seq2seq_val_pred_24h_df)} predictions")
print(f"   48h: {len(seq2seq_val_pred_48h_df)} predictions")

print("\n✓ All predictions generated!")

### 6.3 Evaluate Performance on Validation Set

In [ ]:

# Helper function to evaluate per-county RMSE
def evaluate_per_county(truth, pred_df, locations):
    """Evaluate RMSE per county and return list of RMSEs"""
    rmses = []
    for i, loc in enumerate(locations):
        loc_str = str(loc)
        loc_pred = pred_df[pred_df['location'].astype(str) == loc_str]['pred'].values
        if len(loc_pred) == truth.shape[0]:
            county_rmse = rmse(truth[:, i], loc_pred)
            rmses.append(county_rmse)
        else:
            rmses.append(np.nan)
    return rmses

# Evaluate 24h horizon
print("="*70)
print("VALIDATION EVALUATION - 24H HORIZON")
print("="*70)

sarimax_24h_rmses = evaluate_per_county(val_truth_24h, sarimax_val_pred_24h_df, locations)
seq2seq_24h_rmses = evaluate_per_county(val_truth_24h, seq2seq_val_pred_24h_df, locations)
zero_24h_rmses = [rmse(val_truth_24h[:, i], np.zeros(24)) for i in range(len(locations))]

sarimax_24h_avg = np.nanmean(sarimax_24h_rmses)
seq2seq_24h_avg = np.nanmean(seq2seq_24h_rmses)
zero_24h_avg = np.nanmean(zero_24h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_24h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_24h_avg:.4f}")
print(f"  Zero Baseline: {zero_24h_avg:.4f}")

sarimax_24h_imp = ((zero_24h_avg - sarimax_24h_avg) / zero_24h_avg * 100)
seq2seq_24h_imp = ((zero_24h_avg - seq2seq_24h_avg) / zero_24h_avg * 100)

print(f"\nBest Model for 24h: ", end="")
if sarimax_24h_avg < seq2seq_24h_avg and sarimax_24h_avg < zero_24h_avg:
    print(f"✓ SARIMAX ({sarimax_24h_avg:.4f})")
elif seq2seq_24h_avg < sarimax_24h_avg and seq2seq_24h_avg < zero_24h_avg:
    print(f"✓ Seq2Seq ({seq2seq_24h_avg:.4f})")
else:
    print(f"⚠ Zero Baseline is better ({zero_24h_avg:.4f})")

# Evaluate 48h horizon
print("\n" + "="*70)
print("VALIDATION EVALUATION - 48H HORIZON")
print("="*70)

sarimax_48h_rmses = evaluate_per_county(val_truth_48h, sarimax_val_pred_48h_df, locations)
seq2seq_48h_rmses = evaluate_per_county(val_truth_48h, seq2seq_val_pred_48h_df, locations)
zero_48h_rmses = [rmse(val_truth_48h[:, i], np.zeros(48)) for i in range(len(locations))]

sarimax_48h_avg = np.nanmean(sarimax_48h_rmses)
seq2seq_48h_avg = np.nanmean(seq2seq_48h_rmses)
zero_48h_avg = np.nanmean(zero_48h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_48h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_48h_avg:.4f}")
print(f"  Zero Baseline: {zero_48h_avg:.4f}")

sarimax_48h_imp = ((zero_48h_avg - sarimax_48h_avg) / zero_48h_avg * 100)
seq2seq_48h_imp = ((zero_48h_avg - seq2seq_48h_avg) / zero_48h_avg * 100)

print(f"\nBest Model for 48h: ", end="")
if sarimax_48h_avg < seq2seq_48h_avg and sarimax_48h_avg < zero_48h_avg:
    print(f"✓ SARIMAX ({sarimax_48h_avg:.4f})")
elif seq2seq_48h_avg < sarimax_48h_avg and seq2seq_48h_avg < zero_48h_avg:
    print(f"✓ Seq2Seq ({seq2seq_48h_avg:.4f})")
else:
    print(f"⚠ Zero Baseline is better ({zero_48h_avg:.4f})")

# Log validation metrics to wandb
if WANDB_ENABLED:
    wandb.log({
        "val/sarimax_rmse_24h": sarimax_24h_avg,
        "val/seq2seq_rmse_24h": seq2seq_24h_avg,
        "val/zero_baseline_rmse_24h": zero_24h_avg,
        "val/sarimax_rmse_48h": sarimax_48h_avg,
        "val/seq2seq_rmse_48h": seq2seq_48h_avg,
        "val/zero_baseline_rmse_48h": zero_48h_avg,
    })

### 6.4 Visualize Validation Predictions (Top 5 Counties)

In [ ]:
# Visualize predictions for top 5 counties (24h horizon)
print("Visualizing 24h validation predictions for top 5 counties...\n")

fig, axes = plt.subplots(len(top5_locs), 1, figsize=(15, 4*len(top5_locs)))
if len(top5_locs) == 1:
    axes = [axes]

for plot_idx, loc in enumerate(top5_locs):
    ax = axes[plot_idx]
    loc_str = str(loc)
    loc_idx = locations.index(loc_str)
    
    # Get predictions for this location
    sarimax_pred = sarimax_val_pred_24h_df[sarimax_val_pred_24h_df['location'] == loc_str]['pred'].values
    seq2seq_pred = seq2seq_val_pred_24h_df[seq2seq_val_pred_24h_df['location'] == loc_str]['pred'].values
    
    ax.plot(val_timestamps_24h, val_truth_24h[:, loc_idx], label='Actual', 
            color='black', linewidth=2.5, marker='o', markersize=5)
    ax.plot(val_timestamps_24h, sarimax_pred, label='SARIMAX', 
            alpha=0.7, linewidth=2, marker='s', markersize=4)
    ax.plot(val_timestamps_24h, seq2seq_pred, label='Seq2Seq', 
            alpha=0.7, linewidth=2, marker='^', markersize=4)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Zero Baseline')
    
    ax.set_title(f'24h Validation: County {loc}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Outages')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Showing 24h validation predictions for: {[str(loc) for loc in top5_locs]}")

## 7. Test Set Prediction and Submission Generation (24h and 48h)

Train final models on full training data and evaluate on test sets (24h and 48h).

### 7.1 Train Final Models on Full Training Data

In [ ]:
# Train final SARIMAX models on full training data
print("="*70)
print("TRAINING FINAL SARIMAX MODELS (on full training data)")
print("="*70)

sarimax_final_models = {}
for loc in locations:
    loc_str = str(loc)
    y_full = ds_train.out.sel(location=loc).values.astype(float)
    model = safe_fit_sarimax(y_full, order=SARIMAX_ORDER)
    sarimax_final_models[loc_str] = model
    status = "✓" if model is not None else "✗"
    print(f"  {status} County {loc_str}")

print("\n✓ SARIMAX models trained!")


In [ ]:
# Train final Seq2Seq models for 24h and 48h
print("\n" + "="*70)
print("TRAINING FINAL SEQ2SEQ MODELS (on full training data)")
print("="*70)

# 24h model
print("\n1) Training Seq2Seq for 24h horizon...")
X_final_24h, Y_final_24h, input_dim_final, seq2seq_final_scalers_24h = prepare_seq2seq_data(ds_train, SEQ_LEN, 24)
print(f"   Training samples: {X_final_24h.shape[0]}")

seq2seq_final_model_24h = train_seq2seq(
    X_final_24h, Y_final_24h, input_dim_final, 24,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE
)

# 48h model
print("\n2) Training Seq2Seq for 48h horizon...")
X_final_48h, Y_final_48h, _, seq2seq_final_scalers_48h = prepare_seq2seq_data(ds_train, SEQ_LEN, 48)
print(f"   Training samples: {X_final_48h.shape[0]}")

seq2seq_final_model_48h = train_seq2seq(
    X_final_48h, Y_final_48h, input_dim_final, 48,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE
)

print("\n✓ All final models trained!")

### 7.2 Generate Predictions

In [ ]:
# Generate test predictions using the SAME consolidated functions as validation
print("Generating test set predictions...\n")

# 24h predictions
print("1) Test 24h predictions:")
sarimax_test_24h_df = generate_sarimax_predictions(sarimax_final_models, locations, test_24h_timestamps)
seq2seq_test_24h_df = generate_seq2seq_predictions(
    seq2seq_final_model_24h, ds_train, seq2seq_final_scalers_24h,
    24, test_24h_timestamps, locations
)
print(f"   SARIMAX: {len(sarimax_test_24h_df)} predictions")
print(f"   Seq2Seq: {len(seq2seq_test_24h_df)} predictions")

# 48h predictions
print("\n2) Test 48h predictions:")
sarimax_test_48h_df = generate_sarimax_predictions(sarimax_final_models, locations, test_48h_timestamps)
seq2seq_test_48h_df = generate_seq2seq_predictions(
    seq2seq_final_model_48h, ds_train, seq2seq_final_scalers_48h,
    48, test_48h_timestamps, locations
)
print(f"   SARIMAX: {len(sarimax_test_48h_df)} predictions")
print(f"   Seq2Seq: {len(seq2seq_test_48h_df)} predictions")

print("\n✓ All test predictions generated!")

### 7.3 Evaluate Perofrmance on Test Set (demo only)

The following code is just to verify if your submission files by your model can be evaluated properly or if you followed the format of prediciont file. It is using demo test files, ```test_24h_demo.nc``` and ```test_48h_demo.nc```. Again these files are just randome noise but wtih the right shape and format. So the test results say nothing about the final evaluation.

In [ ]:
# Evaluate 24h test set
print("="*70)
print("TEST EVALUATION - 24H HORIZON")
print("="*70)

sarimax_test_24h_rmses = evaluate_per_county(test_24h_truth, sarimax_test_24h_df, locations)
seq2seq_test_24h_rmses = evaluate_per_county(test_24h_truth, seq2seq_test_24h_df, locations)
zero_test_24h_rmses = [rmse(test_24h_truth[:, i], np.zeros(24)) for i in range(len(locations))]

sarimax_test_24h_avg = np.nanmean(sarimax_test_24h_rmses)
seq2seq_test_24h_avg = np.nanmean(seq2seq_test_24h_rmses)
zero_test_24h_avg = np.nanmean(zero_test_24h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_test_24h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_test_24h_avg:.4f}")
print(f"  Zero Baseline: {zero_test_24h_avg:.4f}")

sarimax_test_24h_imp = ((zero_test_24h_avg - sarimax_test_24h_avg) / zero_test_24h_avg * 100)
seq2seq_test_24h_imp = ((zero_test_24h_avg - seq2seq_test_24h_avg) / zero_test_24h_avg * 100)

# Evaluate 48h test set
print("\n" + "="*70)
print("TEST EVALUATION - 48H HORIZON")
print("="*70)

sarimax_test_48h_rmses = evaluate_per_county(test_48h_truth, sarimax_test_48h_df, locations)
seq2seq_test_48h_rmses = evaluate_per_county(test_48h_truth, seq2seq_test_48h_df, locations)
zero_test_48h_rmses = [rmse(test_48h_truth[:, i], np.zeros(48)) for i in range(len(locations))]

sarimax_test_48h_avg = np.nanmean(sarimax_test_48h_rmses)
seq2seq_test_48h_avg = np.nanmean(seq2seq_test_48h_rmses)
zero_test_48h_avg = np.nanmean(zero_test_48h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_test_48h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_test_48h_avg:.4f}")
print(f"  Zero Baseline: {zero_test_48h_avg:.4f}")

sarimax_test_48h_imp = ((zero_test_48h_avg - sarimax_test_48h_avg) / zero_test_48h_avg * 100)
seq2seq_test_48h_imp = ((zero_test_48h_avg - seq2seq_test_48h_avg) / zero_test_48h_avg * 100)

# Log test metrics to wandb
if WANDB_ENABLED:
    wandb.log({
        "test/sarimax_rmse_24h": sarimax_test_24h_avg,
        "test/seq2seq_rmse_24h": seq2seq_test_24h_avg,
        "test/zero_baseline_rmse_24h": zero_test_24h_avg,
        "test/sarimax_rmse_48h": sarimax_test_48h_avg,
        "test/seq2seq_rmse_48h": seq2seq_test_48h_avg,
        "test/zero_baseline_rmse_48h": zero_test_48h_avg,
    })

### 7.4 Save Predictions to CSV

In [ ]:
# Save predictions to CSV files for submission
print("Saving predictions to CSV files...\n")

# Save SARIMAX predictions
sarimax_test_24h_df.to_csv(os.path.join(RESULTS_DIR, "sarimax_pred_24h.csv"), index=False)
sarimax_test_48h_df.to_csv(os.path.join(RESULTS_DIR, "sarimax_pred_48h.csv"), index=False)
print("✓ SARIMAX predictions saved:")
print(f"  - {os.path.join(RESULTS_DIR, 'sarimax_pred_24h.csv')}")
print(f"  - {os.path.join(RESULTS_DIR, 'sarimax_pred_48h.csv')}")

# Save Seq2Seq predictions
seq2seq_test_24h_df.to_csv(os.path.join(RESULTS_DIR, "seq2seq_pred_24h.csv"), index=False)
seq2seq_test_48h_df.to_csv(os.path.join(RESULTS_DIR, "seq2seq_pred_48h.csv"), index=False)
print("\n✓ Seq2Seq predictions saved:")
print(f"  - {os.path.join(RESULTS_DIR, 'seq2seq_pred_24h.csv')}")
print(f"  - {os.path.join(RESULTS_DIR, 'seq2seq_pred_48h.csv')}")

print("\n" + "="*70)
print("All predictions saved successfully!")
print("="*70)


### 7.5 Evaluate Performance on Test Set

We will be evluating your submitted files following similar procedure in this part but with actual test files for your predictive performance.

You should run these to check your saved files works with these code.

In [ ]:
your_24hr_prediction_filepath = "results/sarimax_pred_24h.csv"
your_48hr_prediction_filepath = "results/sarimax_pred_48h.csv"
# your_24hr_prediction_filepath = "results/seq2seq_pred_24h.csv"
# your_48hr_prediction_filepath = "results/seq2seq_pred_48h.csv"

# load your prediction file
df_24 = pd.read_csv(your_24hr_prediction_filepath)
df_48 = pd.read_csv(your_48hr_prediction_filepath)

In [ ]:
# You prediction file should have three columns: timestamp, location, pred
# and number of rows should be 24*83 = 1992 or 48*83 = 3984 for 24h and 48h prediction respectively

# check the shape
assert df_24.shape == (1992,3)
assert df_48.shape == (3984,3)


# check the column names
assert df_24.columns.tolist() == ['timestamp', 'location', 'pred']
assert df_48.columns.tolist() == ['timestamp', 'location', 'pred']

In [ ]:
# Just making sure, evlauate on test set again.
test_rmses_24 = evaluate_per_county(test_24h_truth, df_24, locations)
test_rmses_48 = evaluate_per_county(test_48h_truth, df_48, locations)
test_rmses_avg_24 = np.nanmean(test_rmses_24)
test_rmses_avg_48 = np.nanmean(test_rmses_48)

print(f"test_24h_rmses: {test_rmses_avg_24}")
print(f"test_48h_rmses: {test_rmses_avg_48}")

## 8. Part II: Backup Generator Pre-positioning

Based on Part I predictions, we allocate **5 backup generators** (each serving up to **1,000 households**) across Michigan counties to maximize outage mitigation.

**Strategy**:
1. **Decision-aware weight search**: Learn the optimal blending weight `w` between 24h and 48h predictions on the validation set, optimizing directly for generator placement effectiveness (not RMSE).
2. **Greedy allocation**: Iteratively assign each generator to the county with the highest marginal outage mitigation.
3. **Evaluate**: Compare our allocation against the oracle (perfect-foresight) allocation.

In [ ]:
# ============================================================
# Part II Helper Functions
# ============================================================

def greedy_allocate_generators(outage_matrix, n_generators, capacity):
    """
    Greedy allocation: at each step, place one generator at the county
    where it would mitigate the most outage-hours.
    
    outage_matrix: (T, L) - hourly predicted outages per county
    Returns: (allocation array, assignment log)
    """
    T, L = outage_matrix.shape
    allocation = np.zeros(L, dtype=int)
    assignment_log = []
    
    for g in range(n_generators):
        best_county = -1
        best_mitigated = -1
        
        for j in range(L):
            current_cap = allocation[j] * capacity
            current_mitigated = np.sum(np.minimum(outage_matrix[:, j], current_cap))
            
            new_cap = (allocation[j] + 1) * capacity
            new_mitigated = np.sum(np.minimum(outage_matrix[:, j], new_cap))
            
            marginal = new_mitigated - current_mitigated
            if marginal > best_mitigated:
                best_mitigated = marginal
                best_county = j
        
        allocation[best_county] += 1
        assignment_log.append({
            "generator": g + 1,
            "county_fips": locations[best_county],
            "marginal_mitigated": best_mitigated,
            "total_at_county": allocation[best_county],
        })
    
    return allocation, assignment_log


def evaluate_allocation(truth_matrix, allocation, capacity):
    """
    Evaluate a generator allocation against ground truth.
    Returns total outage-hours, mitigated hours, and per-county breakdown.
    """
    T, L = truth_matrix.shape
    total_outage_hours = np.sum(truth_matrix)
    mitigated = 0.0
    per_county_results = []
    
    for j in range(L):
        county_cap = allocation[j] * capacity
        county_total = np.sum(truth_matrix[:, j])
        if county_cap > 0:
            county_mitigated = np.sum(np.minimum(truth_matrix[:, j], county_cap))
        else:
            county_mitigated = 0.0
        mitigated += county_mitigated
        if allocation[j] > 0:
            per_county_results.append({
                "county_idx": j,
                "generators": allocation[j],
                "county_outage_hours": county_total,
                "mitigated": county_mitigated,
                "mitigation_rate": county_mitigated / county_total if county_total > 0 else 0,
            })
    
    return {
        "total_outage_hours": total_outage_hours,
        "total_mitigated": mitigated,
        "mitigation_rate": mitigated / total_outage_hours if total_outage_hours > 0 else 0,
        "per_county": per_county_results,
    }

print("✓ Part II helper functions defined: greedy_allocate_generators, evaluate_allocation")

In [ ]:
# ============================================================
# Step 0: Weight Search — Learn optimal 24h/48h blend on validation set
# ============================================================
# The key insight: 24h predictions are more accurate (shorter horizon),
# but 48h predictions cover the full event. By blending them for the
# overlapping first 24 hours, we get a better decision input.
#
# We search for w that maximizes ACTUAL outage mitigation on val truth
# (not RMSE — we optimize the downstream decision directly).

NUM_GENERATORS = 5
GENERATOR_CAPACITY = 1000

# --- Helper: build prediction matrix from a DataFrame ---
def build_pred_matrix(pred_df, horizon, locations):
    """Convert prediction DataFrame to (T, L) numpy matrix."""
    mat = np.zeros((horizon, len(locations)))
    for i, loc in enumerate(locations):
        loc_pred = pred_df[pred_df['location'].astype(str) == str(loc)]['pred'].values
        if len(loc_pred) == horizon:
            mat[:, i] = np.maximum(loc_pred, 0)  # clip negatives
    return mat


def blend_predictions(pred_24h_mat, pred_48h_mat, w):
    """
    Blend 24h and 48h predictions into a single 48h matrix.
    
    Hours  1-24: w * pred_24h + (1-w) * pred_48h[:24]
    Hours 25-48: pred_48h[24:48]
    
    w=1.0 → trust 24h model completely for first 24 hours
    w=0.0 → trust 48h model completely
    """
    blended = pred_48h_mat.copy()
    blended[:24, :] = w * pred_24h_mat + (1 - w) * pred_48h_mat[:24, :]
    return np.maximum(blended, 0)  # ensure non-negative


# --- Build validation prediction matrices ---
# Use whichever model is best; here we use seq2seq for both
val_pred_24h_mat = build_pred_matrix(seq2seq_val_pred_24h_df, 24, locations)
val_pred_48h_mat = build_pred_matrix(seq2seq_val_pred_48h_df, 48, locations)

# Validation truth (48h) — this is what we evaluate against
# We need the full 48h val truth; if val period is shorter, we pad
val_out = ds_val.out.transpose("timestamp", "location").values.astype(float)
val_truth_full = val_out[:48, :] if val_out.shape[0] >= 48 else val_out

print("="*70)
print("WEIGHT SEARCH: Optimizing 24h/48h blend for generator placement")
print("="*70)
print(f"\nValidation prediction shapes:")
print(f"  24h model: {val_pred_24h_mat.shape}")
print(f"  48h model: {val_pred_48h_mat.shape}")
print(f"  Val truth: {val_truth_full.shape}")

# --- Grid search over w ---
w_candidates = np.arange(0, 1.01, 0.05)  # 0.00, 0.05, ..., 1.00
search_results = []

for w in w_candidates:
    blended = blend_predictions(val_pred_24h_mat, val_pred_48h_mat, w)
    
    # Use the blended matrix for allocation (truncate to match truth length)
    alloc_horizon = min(blended.shape[0], val_truth_full.shape[0])
    blended_truncated = blended[:alloc_horizon, :]
    truth_truncated = val_truth_full[:alloc_horizon, :]
    
    allocation, _ = greedy_allocate_generators(blended_truncated, NUM_GENERATORS, GENERATOR_CAPACITY)
    
    # Evaluate on TRUTH
    total_outage = np.sum(truth_truncated)
    mitigated = 0.0
    for j in range(len(locations)):
        cap = allocation[j] * GENERATOR_CAPACITY
        if cap > 0:
            mitigated += np.sum(np.minimum(truth_truncated[:, j], cap))
    
    mitigation_rate = mitigated / total_outage if total_outage > 0 else 0
    
    # Also compute blended prediction RMSE (for reference)
    pred_rmse = float(np.sqrt(np.mean((truth_truncated - blended_truncated[:alloc_horizon])**2)))
    
    search_results.append({
        "w": round(w, 2),
        "mitigated": mitigated,
        "mitigation_rate": mitigation_rate,
        "pred_rmse": pred_rmse,
        "allocation": allocation.copy(),
    })

# Find optimal w
search_df = pd.DataFrame(search_results)
best_idx = search_df['mitigated'].idxmax()
best_w = search_df.loc[best_idx, 'w']
best_mitigated = search_df.loc[best_idx, 'mitigated']
best_rate = search_df.loc[best_idx, 'mitigation_rate']

# Also find w=0 (48h only) and w=1 (24h only) for comparison
w0_result = search_df[search_df['w'] == 0.0].iloc[0]
w1_result = search_df[search_df['w'] == 1.0].iloc[0]

print(f"\n{'─'*70}")
print(f"{'w':>6s} │ {'Mitigated':>12s} │ {'Rate':>8s} │ {'Pred RMSE':>10s} │ Note")
print(f"{'─'*70}")
for _, row in search_df.iterrows():
    note = ""
    if row['w'] == best_w:
        note = "← BEST"
    elif row['w'] == 0.0:
        note = "(48h only)"
    elif row['w'] == 1.0:
        note = "(24h only)"
    print(f"{row['w']:6.2f} │ {row['mitigated']:12.0f} │ {row['mitigation_rate']:7.2%} │ {row['pred_rmse']:10.2f} │ {note}")

print(f"\n✓ Optimal weight: w = {best_w:.2f}")
print(f"  Mitigated: {best_mitigated:.0f} outage-hours ({best_rate:.2%})")
print(f"  vs 48h-only (w=0): {w0_result['mitigated']:.0f} ({w0_result['mitigation_rate']:.2%})")
print(f"  vs 24h-only (w=1): {w1_result['mitigated']:.0f} ({w1_result['mitigation_rate']:.2%})")
improvement_over_48h = (best_mitigated - w0_result['mitigated']) / w0_result['mitigated'] * 100 if w0_result['mitigated'] > 0 else 0
print(f"  Improvement over 48h-only: {improvement_over_48h:+.1f}%")

OPTIMAL_W = best_w

In [ ]:
# --- Weight Search Visualization ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Mitigation vs Weight (the key decision metric)
ax1 = axes[0]
ax1.plot(search_df['w'], search_df['mitigated'], 'o-', color='#2ecc71', 
         linewidth=2, markersize=6, label='Outage-hours mitigated')
ax1.axvline(x=best_w, color='red', linestyle='--', linewidth=2, alpha=0.7,
            label=f'Optimal w={best_w:.2f}')
ax1.axhline(y=w0_result['mitigated'], color='gray', linestyle=':', alpha=0.5,
            label=f'48h-only baseline')
ax1.fill_between(search_df['w'], search_df['mitigated'], alpha=0.1, color='#2ecc71')
ax1.set_xlabel('Weight w  (0=48h only, 1=24h only for first 24h)', fontsize=11)
ax1.set_ylabel('Outage-Hours Mitigated', fontsize=11)
ax1.set_title('Generator Effectiveness vs Blend Weight\n(higher = better)', 
              fontsize=13, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Right: RMSE vs Weight (shows prediction accuracy is different from decision quality)
ax2 = axes[1]
ax2_twin = ax2.twinx()

line1 = ax2.plot(search_df['w'], search_df['mitigation_rate'] * 100, 's-', 
                 color='#2ecc71', linewidth=2, markersize=5, label='Mitigation rate (%)')
line2 = ax2_twin.plot(search_df['w'], search_df['pred_rmse'], '^-', 
                      color='#e74c3c', linewidth=2, markersize=5, label='Prediction RMSE')
ax2.axvline(x=best_w, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax2.set_xlabel('Weight w', fontsize=11)
ax2.set_ylabel('Mitigation Rate (%)', fontsize=11, color='#2ecc71')
ax2_twin.set_ylabel('Prediction RMSE', fontsize=11, color='#e74c3c')
ax2.set_title('Decision Quality vs Prediction Accuracy\n(they may not align!)', 
              fontsize=13, fontweight='bold')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "weight_search.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Weight search visualization saved to results/weight_search.png")

# Log weight search results to wandb
if WANDB_ENABLED:
    # Log the search curve as a wandb table (for interactive exploration)
    ws_table = wandb.Table(
        columns=["w", "mitigated", "mitigation_rate", "pred_rmse"],
        data=[[r['w'], r['mitigated'], r['mitigation_rate'], r['pred_rmse']] 
              for r in search_results]
    )
    wandb.log({
        "weight_search/results_table": ws_table,
        "weight_search/optimal_w": best_w,
        "weight_search/best_mitigated": best_mitigated,
        "weight_search/best_mitigation_rate": best_rate,
        "weight_search/improvement_over_48h_only_pct": improvement_over_48h,
        "weight_search/chart": wandb.Image(
            os.path.join(RESULTS_DIR, "weight_search.png")),
    })
    
    # Also log each w as a separate step for wandb line chart
    for r in search_results:
        wandb.log({
            "weight_search_curve/w": r['w'],
            "weight_search_curve/mitigated": r['mitigated'],
            "weight_search_curve/mitigation_rate": r['mitigation_rate'],
            "weight_search_curve/pred_rmse": r['pred_rmse'],
        })
    
    print("✓ Weight search results logged to W&B")

In [ ]:
# ============================================================
# Part II: Apply optimal weight to TEST predictions & allocate
# ============================================================

# --- Step 1: Build test prediction matrices ---
tracked_per_county = ds_train.tracked.mean(dim="timestamp").values.astype(float)

test_pred_24h_mat = build_pred_matrix(seq2seq_test_24h_df, 24, locations)
test_pred_48h_mat = build_pred_matrix(seq2seq_test_48h_df, 48, locations)

# Apply the optimal weight learned on validation
print("="*70)
print(f"APPLYING OPTIMAL WEIGHT (w={OPTIMAL_W:.2f}) TO TEST PREDICTIONS")
print("="*70)

blended_test_mat = blend_predictions(test_pred_24h_mat, test_pred_48h_mat, OPTIMAL_W)

# Also build pure 48h matrix for comparison
truth_matrix_48h = test_48h_truth

print(f"\nTest prediction shapes:")
print(f"  24h model: {test_pred_24h_mat.shape}")
print(f"  48h model: {test_pred_48h_mat.shape}")
print(f"  Blended (w={OPTIMAL_W:.2f}): {blended_test_mat.shape}")

# --- Step 2: Per-county stats ---
county_stats = []
for i, loc in enumerate(locations):
    pred_peak = float(np.max(blended_test_mat[:, i]))
    pred_total = float(np.sum(blended_test_mat[:, i]))
    tracked_val = float(tracked_per_county[i])
    outage_rate = pred_peak / tracked_val if tracked_val > 0 else 0
    county_stats.append({
        "location": loc, "idx": i, "tracked": tracked_val,
        "pred_peak_outage": pred_peak, "pred_total_outage": pred_total,
        "outage_rate": outage_rate,
    })

stats_df = pd.DataFrame(county_stats).sort_values("pred_total_outage", ascending=False)
print(f"\nTop 15 counties by predicted total outage (blended):")
print(stats_df[["location", "tracked", "pred_peak_outage", "pred_total_outage", "outage_rate"]].head(15).to_string(index=False))


# --- Step 3: Allocate using blended predictions ---
pred_allocation, pred_log = greedy_allocate_generators(
    blended_test_mat, NUM_GENERATORS, GENERATOR_CAPACITY
)

print(f"\n{'='*70}")
print(f"GENERATOR ALLOCATION (blended w={OPTIMAL_W:.2f})")
print(f"{'='*70}")
for entry in pred_log:
    print(f"  Generator {entry['generator']}: County {entry['county_fips']} "
          f"(marginal mitigation: {entry['marginal_mitigated']:.0f} outage-hours, "
          f"total generators there: {entry['total_at_county']})")

recommended_fips = []
for i, count in enumerate(pred_allocation):
    recommended_fips.extend([locations[i]] * count)
print(f"\nRecommended counties (FIPS): {recommended_fips}")

# --- Also compute allocation using ONLY 48h for comparison ---
pred_allocation_48h_only, _ = greedy_allocate_generators(
    test_pred_48h_mat, NUM_GENERATORS, GENERATOR_CAPACITY
)
fips_48h_only = []
for i, count in enumerate(pred_allocation_48h_only):
    fips_48h_only.extend([locations[i]] * count)


# --- Step 4: Evaluate on demo test truth ---
eval_result = evaluate_allocation(truth_matrix_48h, pred_allocation, GENERATOR_CAPACITY)
eval_48h_only = evaluate_allocation(truth_matrix_48h, pred_allocation_48h_only, GENERATOR_CAPACITY)

# Oracle
oracle_allocation, oracle_log = greedy_allocate_generators(
    truth_matrix_48h, NUM_GENERATORS, GENERATOR_CAPACITY
)
oracle_result = evaluate_allocation(truth_matrix_48h, oracle_allocation, GENERATOR_CAPACITY)
oracle_fips = []
for i, count in enumerate(oracle_allocation):
    oracle_fips.extend([locations[i]] * count)

efficiency = eval_result['total_mitigated'] / oracle_result['total_mitigated'] if oracle_result['total_mitigated'] > 0 else 0

print(f"\n{'='*70}")
print("EVALUATION ON DEMO TEST SET (48h truth)")
print(f"{'='*70}")
print(f"\n  {'Strategy':<25s} {'Mitigated':>12s} {'Rate':>8s} {'Counties'}")
print(f"  {'─'*75}")
print(f"  {'Blended (w='+str(OPTIMAL_W)+')':<25s} {eval_result['total_mitigated']:>12.0f} {eval_result['mitigation_rate']:>7.2%}  {recommended_fips}")
print(f"  {'48h-only (w=0)':<25s} {eval_48h_only['total_mitigated']:>12.0f} {eval_48h_only['mitigation_rate']:>7.2%}  {fips_48h_only}")
print(f"  {'Oracle (perfect info)':<25s} {oracle_result['total_mitigated']:>12.0f} {oracle_result['mitigation_rate']:>7.2%}  {oracle_fips}")
print(f"\n  Total outage-hours: {eval_result['total_outage_hours']:.0f}")
print(f"  Decision efficiency (blended / oracle): {efficiency:.2%}")

blend_vs_48h = eval_result['total_mitigated'] - eval_48h_only['total_mitigated']
print(f"  Blending improvement over 48h-only: {blend_vs_48h:+.0f} outage-hours")

print(f"\n  Per-county breakdown (blended allocation):")
for r in eval_result["per_county"]:
    loc = locations[r["county_idx"]]
    print(f"    County {loc}: {r['generators']} gen, "
          f"mitigated {r['mitigated']:.0f}/{r['county_outage_hours']:.0f} "
          f"({r['mitigation_rate']:.1%})")

In [ ]:
# --- Step 5: Visualization ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Top counties with generator assignments
top_n = 15
top_stats = stats_df.head(top_n)

ax1 = axes[0]
colors = ['#e74c3c' if pred_allocation[int(row['idx'])] > 0 else '#3498db' 
          for _, row in top_stats.iterrows()]
bars = ax1.barh(range(top_n), top_stats['pred_total_outage'].values, color=colors)
ax1.set_yticks(range(top_n))
ax1.set_yticklabels([str(loc) for loc in top_stats['location'].values])
ax1.set_xlabel("Predicted Total Outage-Hours (48h, blended)")
ax1.set_title(f"Top 15 Counties by Predicted Outage\n(Red = Generator Assigned, w={OPTIMAL_W:.2f})")
ax1.invert_yaxis()

for idx, (_, row) in enumerate(top_stats.iterrows()):
    gen_count = pred_allocation[int(row['idx'])]
    if gen_count > 0:
        ax1.annotate(f"  {gen_count} gen", 
                     xy=(row['pred_total_outage'], idx),
                     va='center', fontweight='bold', color='#e74c3c')

# Right: 3-way comparison bar chart
ax2 = axes[1]
categories = [f'Blended\n(w={OPTIMAL_W:.2f})', '48h Only\n(w=0)', 'Oracle\n(perfect)']
values = [eval_result['total_mitigated'], eval_48h_only['total_mitigated'], oracle_result['total_mitigated']]
bar_colors = ['#2ecc71', '#3498db', '#f39c12']
bars2 = ax2.bar(categories, values, color=bar_colors, width=0.5)
ax2.set_ylabel("Outage-Hours Mitigated")
ax2.set_title(f"Generator Effectiveness Comparison\n(Total: {eval_result['total_outage_hours']:.0f} outage-hours)")
for bar, val in zip(bars2, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"{val:.0f}\n({val/eval_result['total_outage_hours']:.1%})",
             ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "generator_allocation.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Allocation visualization saved")


# --- Timeline for assigned counties ---
assigned_counties = [(i, loc, pred_allocation[i]) for i, loc in enumerate(locations) if pred_allocation[i] > 0]

if assigned_counties:
    fig, axes_t = plt.subplots(len(assigned_counties), 1, 
                                figsize=(14, 4*len(assigned_counties)), squeeze=False)
    hours = np.arange(48)
    
    for plot_idx, (cidx, loc, ngen) in enumerate(assigned_counties):
        ax = axes_t[plot_idx, 0]
        truth_line = truth_matrix_48h[:, cidx]
        blended_line = blended_test_mat[:, cidx]
        pred_48h_line = test_pred_48h_mat[:, cidx]
        cap_line = np.full(48, ngen * GENERATOR_CAPACITY)
        mitigated_line = np.minimum(truth_line, cap_line)
        
        ax.fill_between(hours, truth_line, alpha=0.2, color='red', label='Actual outage')
        ax.fill_between(hours, mitigated_line, alpha=0.3, color='green', label='Mitigated')
        ax.plot(hours, truth_line, 'r-', linewidth=1.5)
        ax.plot(hours, blended_line, 'b-', linewidth=2, label=f'Blended pred (w={OPTIMAL_W:.2f})')
        ax.plot(hours, pred_48h_line, 'b--', linewidth=1, alpha=0.5, label='48h-only pred')
        ax.axhline(y=ngen * GENERATOR_CAPACITY, color='orange', linestyle=':', 
                   linewidth=2, label=f'Capacity ({ngen}x{GENERATOR_CAPACITY})')
        
        # Mark the 24h boundary
        ax.axvline(x=24, color='gray', linestyle='--', alpha=0.5)
        ax.annotate('24h boundary', xy=(24, ax.get_ylim()[1]*0.9), fontsize=9, color='gray')
        
        county_mitigated = np.sum(mitigated_line)
        county_total = np.sum(truth_line)
        ax.set_title(f"County {loc} — {ngen} gen | "
                     f"Mitigated: {county_mitigated:.0f}/{county_total:.0f} "
                     f"({county_mitigated/county_total:.0%})",
                     fontsize=12, fontweight='bold')
        ax.set_xlabel("Hours")
        ax.set_ylabel("Outage Count")
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "generator_timeline.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Timeline visualization saved")

In [ ]:
# --- Step 6: Save decision file & log to W&B ---

# Save recommended_counties.txt for submission
decision_path = os.path.join(RESULTS_DIR, "recommended_counties.txt")
with open(decision_path, "w") as f:
    f.write(str(recommended_fips))
print(f"✓ Decision file saved: {decision_path}")
print(f"  Content: {recommended_fips}")

# Log all generator allocation results to wandb
if WANDB_ENABLED:
    wandb.log({
        # Weight search
        "generator/optimal_w": OPTIMAL_W,
        # Blended allocation
        "generator/blended_mitigated": eval_result["total_mitigated"],
        "generator/blended_mitigation_rate": eval_result["mitigation_rate"],
        # 48h-only baseline
        "generator/48h_only_mitigated": eval_48h_only["total_mitigated"],
        "generator/48h_only_mitigation_rate": eval_48h_only["mitigation_rate"],
        # Oracle
        "generator/oracle_mitigated": oracle_result["total_mitigated"],
        "generator/oracle_mitigation_rate": oracle_result["mitigation_rate"],
        # Efficiency
        "generator/decision_efficiency": efficiency,
        "generator/blend_improvement_over_48h": blend_vs_48h,
        "generator/total_outage_hours": eval_result["total_outage_hours"],
    })
    
    # Summary table: all three strategies
    summary_table = wandb.Table(
        columns=["Strategy", "Counties", "Mitigated", "Rate", "vs Oracle"],
        data=[
            [f"Blended (w={OPTIMAL_W})", str(recommended_fips), 
             eval_result['total_mitigated'], f"{eval_result['mitigation_rate']:.2%}", f"{efficiency:.1%}"],
            ["48h-only", str(fips_48h_only),
             eval_48h_only['total_mitigated'], f"{eval_48h_only['mitigation_rate']:.2%}", 
             f"{eval_48h_only['total_mitigated']/oracle_result['total_mitigated']:.1%}" if oracle_result['total_mitigated'] > 0 else "N/A"],
            ["Oracle", str(oracle_fips),
             oracle_result['total_mitigated'], f"{oracle_result['mitigation_rate']:.2%}", "100%"],
        ]
    )
    wandb.log({"generator/strategy_comparison": summary_table})
    
    # Log images
    for fname, key in [("generator_allocation.png", "allocation_chart"),
                       ("generator_timeline.png", "timeline_chart"),
                       ("weight_search.png", "weight_search_chart")]:
        fpath = os.path.join(RESULTS_DIR, fname)
        if os.path.exists(fpath):
            wandb.log({f"generator/{key}": wandb.Image(fpath)})
    
    print("✓ All generator metrics and visualizations logged to W&B")

print(f"\n{'='*70}")
print("PART II COMPLETE")
print(f"{'='*70}")
print(f"  Optimal weight: w = {OPTIMAL_W:.2f}")
print(f"  Recommended counties: {recommended_fips}")
print(f"  Decision file: {decision_path}")
print(f"  Decision efficiency: {efficiency:.2%}")

In [ ]:
# ============================================================
# Finish W&B run
# ============================================================
if WANDB_ENABLED:
    wandb.finish()
    print("✓ W&B run finished. Check your dashboard for all results.")
else:
    print("W&B was not enabled for this run.")